# 卷积神经网络

在本次课程中，我们将继续神经网络的学习，但是这一次，我们学习的是一种全新的神经网络-卷积神经网络（CNN），我们将逐渐深入了解 CNN 的卷积、池化操作的概念和目的，理解并学会使用这个非常重要的神经网络模型。

## 1. CNN 的工作原理


### 1.1 引言：图像识别的挑战与CNN的崛起

我们如何让计算机像人一样“看懂”一张图片？传统的图像处理方法在面对复杂多变的图像时，常常力不从心，例如如何区分一只猫和一只狗？如何识别图片中特定物体的位置？

*   **传统方法的局限：** 传统方法往往需要人工设计复杂的特征提取器，耗时耗力，且泛化能力差。
*   **深度学习的曙光：** 神经网络的出现为图像识别带来了新的希望，而卷积神经网络（CNN）更是其中的佼佼者，它能够自动从图像中学习和提取有用的特征。
*   **为什么需要CNN？**

CNN 十分擅长图像处理工作。你可能会问，为什么我们之前学过的全连接神经网络不能用来处理图像呢？

答案是：可以，但没必要。

在图片处理工作中，我们把组成图片的每个像素点的信息当做输入层 (input layer)，对于黑白像素，可以用 0 和 1 来表示，对于彩色像素，可以用 RGB 值来表示。

对于全连接神经网络，我们要将所有的像素点都输入神经网络，然后构建一个庞大的神经网络模型。

![](https://imgbed.momodel.cn/20201030144651.png)

如上图所示，第一层的神经元可能只能识别不同的颜色，第二层的神经元可能可以识别特定的纹理，再往后的某一层神经元也许就能识别一个复杂的物体，比如蜂巢或者汽车轮胎。

但问题是，如果我们使用全连接神经网络，一张 $100 \times 100$ 的彩色图片，光输入层就有 $100 \times 100 \times 3=30000$ 个参数，如果是整个神经网络，这就太庞大了。

如果我们仔细想想，我们真的需要把每个像素的信息输入神经网络吗？

<div align=left>
<video src="https://imgbed.momodel.cn/CNN1.mp4" controls="controls" width=700/>
</div>

基于我们的认知，我们会发现一些事物都有自己的特点（或者称为 pattern），比如含有一只鸟的图片，我们也许不需要知道这只鸟是在飞还是停在树上，又或者是什么颜色，只要它是只鸟，它的鸟嘴就会符合一定的 pattern，我们可以根据这点，用一个神经元专门去侦测这种 pattern，而不需要知道图片里所有的像素信息，这样，神经网络的大小就可以大大缩小。

这样做的另一个好处是我们可以复用这个 pattern，比如说鸟嘴可能出现在图片里不同的地方，我们不需要用多个神经元去匹配不同位置的鸟嘴，我们只需要用一个神经元去匹配不同位置的鸟嘴就行，这样就减少了神经元的重复。

### 1.2 局部感知与权值共享：CNN的核心思想

*   **局部感知：** 就像人眼在观察物体时，会先关注局部特征（如眼睛、鼻子），然后将这些局部信息组合起来形成对整体的认知。CNN也效仿了这一点，它的“神经元”只关注输入图像的一个局部区域。
    *   **类比：** 你正在看一幅画，你的眼睛不是一次性扫描整幅画，而是聚焦在画面的某个小区域，然后移动到下一个区域。
*   **权值共享：** 如果一个特征（比如垂直边缘）在图像的不同位置都可能出现，我们不需要为每个位置都训练一个独立的检测器。CNN通过“权值共享”机制，让同一个特征检测器（卷积核）在图像的不同区域重复使用。
    *   **类比：** 就像一个万能的“滤镜”，可以应用到图像的任何地方来检测特定的纹理或形状。这大大减少了模型的参数数量，提高了训练效率。

```mermaid
flowchart TD
    A[输入图像] --> B{定义卷积核K};
    subgraph 卷积操作
        B --滑动--> C[局部区域1];
        B --滑动--> D[局部区域2];
        B --滑动--> E[局部区域N];
    end
    C --> F[提取特征F1];
    D --> G[提取特征F2];
    E --> H[提取特征FN];
    F & G & H --> I[构建特征图];

    style A fill:#e3f2fd,stroke:#333,stroke-width:2px;
    style B fill:#fff3e0,stroke:#f57c00,stroke-width:2px;
    style C fill:#e0f2f7,stroke:#0288d1,stroke-width:1px;
    style D fill:#e0f2f7,stroke:#0288d1,stroke-width:1px;
    style E fill:#e0f2f7,stroke:#0288d1,stroke-width:1px;
    style F fill:#e8f5e8,stroke:#43a047,stroke-width:1px;
    style G fill:#e8f5e8,stroke:#43a047,stroke-width:1px;
    style H fill:#e8f5e8,stroke:#43a047,stroke-width:1px;
    style I fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px;

    classDef note fill:#fff,stroke:#f57c00,stroke-width:2px,color:#f57c00;
    classDef noteText fill:#fff,stroke:#f57c00,stroke-width:2px,color:#f57c00;
    note1(权值共享) --> B;
    class note1 note;
    style note1 fill:#fff,stroke:#f57c00,stroke-width:2px,color:#f57c00;
```

**图表说明:**
*   **局部感知:** 卷积核`K`在输入图像上通过“滑动”的方式依次处理不同的“局部区域”，模拟人眼聚焦局部。
*   **权值共享:** 相同的卷积核`K`被“应用于”所有这些局部区域，说明其参数是共享的，大大减少了模型复杂度。
*   最终，从每个局部区域提取的特征组合起来，构建出完整的“特征图”。


### 1.3 CNN 整体架构：从输入到输出的旅程
<div align=left>
<video src="https://imgbed.momodel.cn/CNN2.mp4" controls="controls" width=700/>
</div>
CNN 的基本结构如图所示：


<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/20201030151428.png" width="600px"/></div>
</div>


*   **完整流程示意图：**
    1.  **输入层：** 接收原始图像数据。
    2.  **卷积层：** 使用多个卷积核提取图像的局部特征，生成特征图。
    3.  **池化层：** 对特征图进行降采样，减少维度和计算量，并增强鲁棒性。
    4.  **重复：** 通常会重复多个卷积层和池化层的组合，以提取越来越抽象、高级的特征。
    5.  **展平层 (Flatten)：** 将多维的特征图展平为一维向量，以便输入到全连接层。
    6.  **全连接层：** 将提取到的高级特征组合起来，进行最终的分类或回归决策。
    7.  **输出层：** 输出最终的预测结果（例如，图像所属的类别）。
*   **总结：** 总结各层级在整个网络中的作用和配合，强调CNN如何从像素级别的信息逐步抽象出高层语义特征。

**CNN 整体架构**

```mermaid
graph LR
    A[输入图像] --> B(卷积层1);
    B --> C(池化层1);
    C --> D(卷积层2);
    D --> E(池化层2);
    E --> I(...);
    I --> F(展平层);
    F --> G(全连接层);
    G --> H(输出层 - 分类结果);

    style A fill:#e3f2fd,stroke:#333,stroke-width:2px;
    style B fill:#fff3e0,stroke:#f57c00,stroke-width:2px;
    style C fill:#fff3e0,stroke:#f57c00,stroke-width:2px;
    style D fill:#fff3e0,stroke:#f57c00,stroke-width:2px;
    style E fill:#fff3e0,stroke:#f57c00,stroke-width:2px;
    style F fill:#e0f2f7,stroke:#0288d1,stroke-width:2px;
    style G fill:#e8f5e8,stroke:#43a047,stroke-width:2px;
    style H fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px;
```
一张图片会由很多个像素 (Pixel) 组成的矩阵表示，作为输入层。

经过中间层的卷积 (Convolution) 和最大池化 (Max Pooling)，我们会得到一个高维的矩阵，这一过程可以重复多次，用以提取更高维度的特征。


然后我们将这个矩阵进行 Flatten 操作，把高维的矩阵 “压平”，将结果输入全连接神经网络中，最后得出结果。

### 1.4 卷积层：特征提取的“滤镜”
卷积层是CNN的心脏，它通过卷积操作从输入图像中提取各种特征。
 
我们可以把卷积操作想象成给图像打“滤镜”。不同的滤镜（卷积核）能提取出不同的信息，比如边缘、纹理、颜色模式等。

*   **卷积核 (Filter/Kernel)：** 这是一个小型的矩阵，它在图像上滑动，与图像的局部区域进行逐元素相乘并求和，得到一个新的像素值。这个新的像素值代表了该局部区域的某种特征强度。

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250905154221981.gif" width="400px"/></div>
</div>

*   **特征图 (Feature Map)：** 卷积操作的输出就是特征图。每个特征图都对应一个特定的卷积核所提取的特征。一个卷积层通常会包含多个卷积核，从而提取出多种不同的特征。

你可能会对许多概念感到迷惑，卷积到底是什么，filter 又是什么？别急，我们慢慢来。

下图中左侧是一张图片，我们用 0 和 1 来表示图片的黑白像素，大小为 $6 \times 6$ 的矩阵。

右侧有两个不同的 filter，他们是大小为 3*3 的矩阵，通过训练得到，我们可以用他们来侦测不同的 pattern，也就是图片的某个特征。

![](https://imgbed.momodel.cn/20201103175911.png)

如何侦测呢，我们只需要用这个 filter 和图片矩阵中的某一个部分（3*3 的矩阵）做内积，就可以侦测这一部分的 pattern，然后按照一定的步长移动 filter，就可以获得图片中所有位置含有这个 pattern的状况。 

<div align=left>
<video src="https://imgbed.momodel.cn/CNN3.mp4" controls="controls" width=700/>
</div>


刚刚我们的图片是黑白的，如果是彩色的图片呢？

我们通常用 RGB 来表示彩色图片，比如，一张图片我们可以用 6 * 6 * 3 的矩阵来表示，这时候我们的 filter 也要做出相应的变化，我们用 3 * 3 * 3 大小的 filter 去侦测 pattern，如下图所示：

![](https://imgbed.momodel.cn/20201103181532.png)

我们上面做的事情就是卷积，简单来说就是提取图片的某一些特征。那么在神经网络中，我们该怎么去实现呢？

<div align=left>
<video src="https://imgbed.momodel.cn/CNN4.mp4" controls="controls" width=700/>
</div>

在全连接神经网络中，我们用上一层的输出乘以每条边的权重（weight），就得到了下一层的输入值。

我们首先把图片矩阵和 filter 矩阵都展平，变成一个一维的向量，对于这个位置的 filter，我们可以把 filter 中的每个值看成是与对应节点的边的 weight，不参与此次 filter 的节点相连的边的 weight 可以看成 0， filter 之后得到的结果就是卷积层神经元的值。

![](https://imgbed.momodel.cn/20201103184540.png)


### 1.5 池化层：特征的“压缩与精炼”
池化是对特征图进行的一种压缩操作，通过在一个小的局部区域内进行汇总统计，用一个值来代表这个区域的特征信息，常用于卷积神经网络（CNN）中。

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250905155603027.webp" width="500px"/></div>
</div>



#### 1.5.1 作用

- 提取代表性信息的同时降低特征维度，具有平移不变性。

- 减少数据维度，降低计算量，提高模型训练速度，同时还能在一定程度上防止过拟合，使模型具有更好的泛化能力，提取更具代表性的特征。

#### 1.5.2 常见类型

- 最大池化（Max Pooling）：在每个池化窗口中，取窗口内的最大值作为该区域的输出值，能突出图像中的显著特征。

<div align=left>
<video src="https://imgbed.momodel.cn/CNN5.mp4" controls="controls" width=700/>
</div>

- 平均池化（Average Pooling）：计算每个池化窗口内元素的平均值作为输出值，能保留特征的整体统计信息。
- 随机池化（Stochastic Pooling）：根据一定的概率分布对池化窗口内的元素进行采样，在训练过程中引入了随机性，有助于提高模型的鲁棒性。

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250905155311694.jpeg" width="800px"/></div>
</div>


#### 1.5.3 最大池化展示

<div class="insertContainerBox" align=center>
<video src="https://imgbed.momodel.cn/CNN5.mp4" controls="controls" width="100%"/>
</div>


max pooling 做的事情就是 sub-sampling，比如，我们可以吧 convolution 之后的结果 4 个一组分组，然后从中选出值最大的那个，这样我们可以让图片中的特征更加突出。

convolution + max pooling 这一过程我们可以执行多次，每次我们都会得到一张比之前更小的 image。

### 1.6 展平


在经过一系列卷积和池化操作后，我们得到了一组高度抽象化的特征图（Feature Maps）。这些特征图是三维的立方体，它们包含了识别目标物体最关键的“线索”。



然而，我们接下来要使用的全连接层（Fully Connected Layer） 有一个特点：它要求输入必须是一个一维的向量（就像一个长长的数组）。这就好比全连接层是一位只看得懂清单的“首席裁判”，而卷积层交给它的是一本立体的“线索相册”。

![](https://imgbed.momodel.cn/20201105101356.png)

展平（Flatten） 操作，就是这位负责把“相册”整理成“清单”的秘书。它的工作非常简单粗暴：

>将多维的特征图（高度 × 宽度 × 通道数），按顺序“压扁”并拼接成一个一维的长向量。

假设卷积网络的最后一层输出了 4 个 大小为 4x4 的特征图。

- 展平前： 数据的形状是 (4, 4, 4)（高、宽、通道）。你可以想象成4张叠在一起的 4x4 方格纸。
- 展平操作：
    - 拿起第一张方格纸（第一个通道），按行（或按列）读取所有数字，得到 16 个数字。
    - 接着拿起第二张方格纸，同样读取 16 个数字，接在第一张的 16 个数字后面。
    - 重复这个过程，直到处理完所有 4 张方格纸。
- 展平后： 我们得到了一个长度为 4 x 4 x 4 = 64 的一维向量。它的形状是 (64,)。

这个 64 维的向量，每一个数字都代表了原始图像中的某种高级特征（比如“是否检测到猫胡须”、“是否有圆形轮廓”等）。这个向量就是全连接层的输入。
    



### 1.7 全连接层
现在，所有的高级特征都被整理成了一个整齐的、一维的“证据清单”。接下来，就可以将这份清单交给“裁判”——全连接层，由它来学习这些特征之间的复杂组合关系。

*   **连接方式：** 全连接层中的每个神经元都与前一层的所有神经元相连接，就像传统神经网络一样。

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250905155713533.webp" width="600px"/></div>
</div>

*   **分类输出：** 最终的全连接层通常会连接到一个输出层（例如，使用 Softmax 激活函数进行多分类），将提取到的特征映射到最终的分类结果（例如，“猫”、“狗”、“汽车”等）。






### 1.8 CNN 代码示例
下面是一个简单的 RNN 实现示例

In [1]:
import tensorflow as tf
from tensorflow.keras import layers

# 创建简单的CNN模型
model = tf.keras.Sequential([
    # 卷积层 + 池化层
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    
    # 展平层
    layers.Flatten(),
    
    # 全连接层
    layers.Dense(10, activation='softmax')
])

# 编译模型
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 打印模型结构
model.summary()

2025-09-10 11:03:58.856927: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-10 11:04:00.491391: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-10 11:04:01.103205: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-10 11:04:02.305547: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-10 11:04:02.480761: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 5408)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        54,090 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 54,410 (212.54 KB)

 Trainable params: 54,410 (212.54 KB)

 Non-trainable params: 0 (0.00 B)

这是一个最简单的 CNN 模型，包含：
- 1 个卷积层（32 个滤波器）
- 1 个池化层
- 1 个全连接输出层

适用于 28x28 像素的单通道图像分类（如 MNIST 数据集）。

## 2. CNN 的应用案例


### 2.1 引言：CNN在现实世界中的魔力

卷积神经网络（CNN） 的出现而改变。它就像给计算机装上了一双真正的“智慧之眼”，让它不仅能“看到”图像，更能“理解”图像的内容。从我们每天使用的手机拍照美颜，到关系国计民生的自动驾驶、医疗诊断， CNN 的身影无处不在，它正在静悄悄地改变我们生活的每一个角落。

- 左上： 智能手机拍照界面，突出显示“人像模式”的虚化效果。
- 右上： 智能安防监控画面，摄像头图标旁有一个绿色的检测框，锁定了一个行人。
- 左下： 自动驾驶汽车的视角，车辆识别出了前方的车辆、行人和道路标志。
- 右下： 一张精美的艺术画，但风格是梵高的《星月夜》，这是风格迁移的效果。
<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250905165607577.png" width="800px"/></div>
</div>


### 2.2 图像分类：识别万物

图像分类是CNN最基础也是最核心的应用。它的任务是回答一个简单的问题：“这张图片里是什么？”

案例：
- 生活趣味： 汽车识别。
- 基础入门： 手写数字识别（MNIST数据集，CNN的“Hello World”）。
- 商业应用： 电商平台的商品自动分类（上传一张鞋子图片，自动归入“女鞋”类目）。
- 农业生产： 通过拍摄叶片照片，识别柑橘黄龙病、黄瓜白粉病等，帮助农民及早防治。

**CNN如何工作？**          
它不是像我们一样一眼就看出这辆车是什么样的，而是通过多层卷积和池化操作，像剥洋葱一样一层层提取特征。

- 底层特征： 第一层可能只识别出一些简单的边缘、角点。
- 中层特征： 更深一些的层，能将边缘组合成纹理、物件的部分（例如车轮）。
- 高层特征： 最后几层，就能组合出完整的物体部件乃至整个物体（一张猫脸）。
- 最终，通过这些提取到的特征，全连接层像一个裁判一样，判断这些特征最符合哪个类别，并输出一个概率（比如：98% 是汽车，2% 是自行车）。


比如在分辨一幅分辨率为 100 万像素的汽车图像时，会先从图像中提取汽车边缘特征，然后生成汽车的部件（如轮子、车门等），最后得到更高层的模式。

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/202501101359499.png" width="800px"/></div>
</div>

### 2.3 目标检测：定位图像中的“他”和“它”

分类告诉我们“是什么”，但很多时候，我们更关心“在哪里？有多少个？”这就是目标检测的任务。

- 自动驾驶： 这是目标检测的“终极考场”！车辆必须实时检测并定位出周围的车辆、行人、交通标志、车道线，一个错误都可能造成严重后果。
- 安防监控： 在机场、火车站等公共场所，自动检测异常行为（如人员聚集、摔倒、遗留可疑物体），并发出警报。
- 工业质检： 在生产线上，用摄像头快速扫描产品（如手机屏幕、芯片），精准定位出划痕、破损等缺陷，效率远超人工。


**CNN 如何实现定位？**
- 主流方法（如 YOLO, SSD）非常聪明：它们不再对整张图只看一次，而是将其分成多个网格，每个网格都负责预测其中是否包含目标物体，以及物体的边界框（Bounding Box）。
>简而言之：“分而治之，同时预测”。 它在提取特征的同时，就完成了对物体位置的判断。

<div class='insertContainerBox column'>
<div align=center><img src="https://imgbed.momodel.cn/202502110908578.png" width="800px"/></div>
</div>

### 2.4 其他应用简述：更多可能

 CNN的能力远不止于此，它还在不断开拓新的疆域。
 
>人脸识别： 不仅是“有没有脸”，更是“这是谁的脸？”（特征提取+比对）。用于手机解锁、移动支付、门禁系统。

>医学影像分析： CNN可以成为医生的“超级助手”，在CT、MRI、X光片中辅助诊断，标注出疑似肿瘤、出血点或骨折部位，提高诊断的准确性和效率。

>图像风格迁移： 将一张照片的内容和另一张名画的风格完美融合，“让你的照片拥有梵高的笔触”。这背后是CNN对图像内容和风格的分离与再合成。

>图像生成： 最新的AI绘画（如DALL-E, Midjourney）、换脸等技术，也大多以CNN或其变体为核心，它们可以从一段文字描述或一个随机噪声中，“无中生有”地创造出逼真的图像。
